# Self-RAG: Retrieval with Quality Grading

Standard RAG pipelines pass retrieved documents to the LLM unconditionally. When the retriever returns low-quality results, the LLM either fabricates an answer or says it cannot help — and neither outcome is visible to the caller. Self-RAG introduces an explicit quality loop: retrieved documents are graded for relevance before generation, generated answers are checked for grounding after generation, and the pipeline retries with a rewritten query if either check fails. This makes quality failures visible and recoverable rather than silent.

**What you'll build:** A self-grading RAG pipeline that evaluates document relevance, generates an answer, checks whether that answer is supported by the retrieved context, and retries up to N times when grading fails.

**Time:** ~25 min | **Difficulty:** Advanced

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide
- Understanding of evaluation metrics and LLM-as-judge patterns
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- How `SelfRAGPipeline` implements the relevance and hallucination grading loop
- How to configure retry behaviour and grading thresholds
- How to read grading decisions from the pipeline's response metadata
- When to use Self-RAG and what it cannot guard against

In [ ]:
!pip install synapsekit -q

## Environment Setup

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import SelfRAGPipeline

llm = OpenAILLM(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Load documents

In [ ]:
docs = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall including its antenna.",
    "Paris is the capital of France and has a population of approximately 2.1 million in the city proper.",
    "The Louvre is the world's most visited art museum, located on the Right Bank of the Seine in Paris.",
    "French is the official language of France and one of the six official languages of the United Nations.",
    "The French Revolution began in 1789 with the storming of the Bastille on July 14th.",
]

await vectorstore.aadd(docs)
print(f"Added {len(docs)} documents to the vector store.")

## Step 3: Configure the SelfRAGPipeline

The grading thresholds are the most important configuration decision. A relevance threshold of 0.7 means a retrieved document must score 0.7 or higher on the 0-1 relevance scale to be included in the generation context. A hallucination threshold of 0.8 means the answer must be grounded in the context at 80% confidence or higher, otherwise a retry is triggered.

In [ ]:
pipeline = SelfRAGPipeline(
    llm=llm,
    vectorstore=vectorstore,
    embeddings=embeddings,
    relevance_threshold=0.7,       # documents below this score are filtered out before generation
    hallucination_threshold=0.8,   # answers below this grounding score trigger a retry
    max_retries=3,                 # maximum number of query-rewrite + regeneration cycles
    top_k=6,                       # retrieve more candidates so filtering has room to work
)
print("SelfRAGPipeline configured.")

## Step 4: Run a query and inspect grading decisions

In [ ]:
async def graded_query():
    result = await pipeline.aquery("How tall is the Eiffel Tower?")
    print(f"Answer: {result.answer}\n")
    print(f"Retries: {result.num_retries}")
    print(f"Hallucination score: {result.hallucination_score:.2f}")
    print(f"\nDocument grades:")
    for doc, grade in result.document_grades:
        status = "INCLUDED" if grade.score >= 0.7 else "FILTERED"
        print(f"  [{status} {grade.score:.2f}] {doc[:70]}")

asyncio.run(graded_query())

## Step 5: Trigger a retry by asking about something not in the documents

This demonstrates the retry loop. The pipeline will attempt to retrieve relevant context, fail the relevance grade, rewrite the query, and try again. After `max_retries` attempts it returns the best answer it found along with a low confidence score.

In [ ]:
async def missing_context_query():
    result = await pipeline.aquery("What is the population of Lyon?")
    print(f"Answer: {result.answer}")
    print(f"Retries: {result.num_retries}")
    print(f"Hallucination score: {result.hallucination_score:.2f}")
    print(f"Confidence: {result.confidence}")  # 'low', 'medium', or 'high'

asyncio.run(missing_context_query())

## Step 6: Stream a graded answer

When streaming, grading happens before the first token is yielded. The `astream` method yields an initial metadata chunk followed by the answer tokens.

In [ ]:
async def stream_graded():
    question = "When did the French Revolution begin?"
    print(f"Q: {question}\n")
    async for chunk in pipeline.astream(question):
        if chunk.is_metadata:
            print(f"[grade: {chunk.hallucination_score:.2f}, retries: {chunk.num_retries}]")
        else:
            print(chunk.text, end="", flush=True)
    print()

asyncio.run(stream_graded())

## Step 7: Use a stronger grader model

The default grader uses the same LLM as generation. For higher-stakes use cases, use a more capable model specifically for grading while keeping generation on the cheaper model.

In [ ]:
from synapsekit.llms.openai import OpenAILLM

grader_llm = OpenAILLM(model="gpt-4o")

pipeline_strong_grader = SelfRAGPipeline(
    llm=llm,               # generation model — cost-efficient
    grader_llm=grader_llm, # grading model — higher accuracy
    vectorstore=vectorstore,
    embeddings=embeddings,
    relevance_threshold=0.7,
    hallucination_threshold=0.8,
    max_retries=3,
    top_k=6,
)
print("Pipeline with strong grader model configured.")

## Complete Working Example

A single self-contained cell that runs the entire Self-RAG pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.pipelines import SelfRAGPipeline

DOCS = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall including its antenna.",
    "Paris is the capital of France and has a population of approximately 2.1 million in the city proper.",
    "The Louvre is the world's most visited art museum, located on the Right Bank of the Seine in Paris.",
    "French is the official language of France and one of the six official languages of the United Nations.",
    "The French Revolution began in 1789 with the storming of the Bastille on July 14th.",
]


async def main():
    llm = OpenAILLM(model="gpt-4o-mini")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    await vectorstore.aadd(DOCS)

    pipeline = SelfRAGPipeline(
        llm=llm,
        vectorstore=vectorstore,
        embeddings=embeddings,
        relevance_threshold=0.7,
        hallucination_threshold=0.8,
        max_retries=3,
        top_k=6,
    )

    questions = [
        "How tall is the Eiffel Tower?",
        "When did the French Revolution begin?",
        "What is the population of Lyon?",  # not in documents — will trigger retries
    ]

    for question in questions:
        result = await pipeline.aquery(question)
        print(f"Q: {question}")
        print(f"A: {result.answer}")
        print(f"   Confidence: {result.confidence} | Hallucination score: {result.hallucination_score:.2f} | Retries: {result.num_retries}\n")


asyncio.run(main())

## Next Steps

- [RAG Fusion](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/rag-fusion) — generate better candidate documents to grade in the first place
- [Cross-Encoder Reranking](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/cross-encoder-reranking) — rerank before grading to reduce the grader's workload
- [Adaptive RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/adaptive-rag) — route queries by complexity so Self-RAG's grading loop is only used for difficult questions